In [12]:
import glob
import os
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, CLIPModel

In [2]:
folder_path = r"/home/aatman/Aatman/Tattle/tattle-research/brainrot/main_analysis_2/visualisation/video_reels"
video_files = glob.glob(os.path.join(folder_path, "*.mp4"))
print(f"Total videos found: {len(video_files)}")

Total videos found: 1472


In [3]:
# pkl_path = "video_data.pkl"
# if os.path.exists(pkl_path):
#     with open(pkl_path, "rb") as f:
#         video_data = pickle.load(f)
#     existing_video_paths = set(entry["video_path"] for entry in video_data)
# else:
#     video_data = []
#     existing_video_paths = set()

# new_video_files = [vf for vf in video_files if vf not in existing_video_paths]
# print(f"New videos to process: {len(new_video_files)}")

In [4]:
def find_i_frames(fname):
    folder, _ = os.path.splitext(os.path.basename(fname))
    folder = (
        folder.replace("(", "_").replace(")", "_").replace(" ", "_").replace("&", "_")
    )
    cmd = f"""mkdir {folder}
    mkdir {folder}/I_frames
    ffmpeg -i "{fname}" -vf "select=eq(pict_type\,I)" -vsync vfr "{folder}/I_frames/frame_%03d.jpg"
    """
    # process = subprocess.Popen(cmd, shell=True)
    process = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    process.wait()
    path = f"{folder}/I_frames"
    frames = []
    for filename in os.listdir(path):
        if filename.endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(path, filename)
            frames.append(Image.open(image_path))
    shutil.rmtree(f"{folder}")
    return frames

In [5]:
%%time
%%capture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CPU times: user 758 ms, sys: 523 ms, total: 1.28 s
Wall time: 5.29 s


In [6]:
def generate_embedding(frames):
    res = []
    for img in frames:
        inputs = processor(images=img, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            embedding = model.get_image_features(**inputs)
            embedding = embedding.transpose(0, 1)
            res.append(
                embedding.detach().cpu().numpy()
            )  # Move tensor to CPU before converting to NumPy
    res = np.hstack(res)
    return res.mean(axis=1).tolist()

In [7]:
def encode_video(fname):
    frames = find_i_frames(fname)
    thumbnail = frames[0]
    embedding = generate_embedding(frames)
    return embedding, thumbnail

In [8]:
video_files = glob.glob(os.path.join(folder_path, "*.mp4"))
print(len(video_files))

1472


In [9]:
# %%time
# embeddings = []
# thumbnails = []

# st = time.time()

# for file in tqdm(video_files):
#     try:
#         embedding, thumbnail = encode_video(file)
#         embeddings.append(embedding)
#         thumbnails.append(thumbnail)
#     except Exception as e:
#         print(f"Error processing {file}: {e}")

# end = time.time()

# time_taken = (end - st) / 60  # in minutes
# print(f"Time taken: {time_taken:.2f}")

In [10]:
# # Create a structured data format
# video_data = []

# # Use tqdm to add a progress bar
# for i, (file, embedding, thumbnail) in tqdm(
#     enumerate(zip(video_files, embeddings, thumbnails)),
#     desc="Processing Videos",
#     total=len(video_files),
# ):
#     filename = os.path.splitext(os.path.basename(file))[0]
#     thumbnail_path = f"thumbnails/{filename}.jpg"
#     os.makedirs("thumbnails", exist_ok=True)
#     thumbnail.save(thumbnail_path)

#     # Store metadata and embedding together
#     video_data.append(
#         {"video_id": filename, "video_path": file, "embedding": embedding, "thumbnail_path": thumbnail_path}
#     )

### new logic to extract and store
- because the old one mis-matched the thumnails

In [11]:
thumbnails_dir = "thumbnails"
os.makedirs(thumbnails_dir, exist_ok=True)

In [14]:
%%time
metadata = []

st = time.time()

for file in tqdm(video_files):
    try:
        embedding, thumbnail = encode_video(file)

        # Save Thumbnail as JPG
        filename = os.path.splitext(os.path.basename(file))[0]
        thumbnail_filename = f"{filename}.jpg"
        thumbnail_path = os.path.join(thumbnails_dir, thumbnail_filename)
        thumbnail.save(thumbnail_path)

        # Append Metadata
        metadata.append({
            "video_id": filename,
            "video_path": file,
            "embedding": embedding,
            "thumbnail_path": thumbnail_path
        })

    except Exception as e:
        print(f"Error processing {file}: {e}")

end = time.time()
time_taken = (end - st) / 60  # in minutes
print(f"Time taken: {time_taken:.2f}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1472/1472 [49:12<00:00,  2.01s/it]

Time taken: 49.21
CPU times: user 1h 15min 41s, sys: 6.08 s, total: 1h 15min 47s
Wall time: 49min 12s


In [16]:
# Save the structured data with pickle
with open("video_data_2.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [15]:
# # Optionally, also save just the embeddings as numpy array for faster loading
# # when you only need the embeddings
# embeddings_array = np.array(embeddings)
# np.save("video_embeddings.npy", embeddings_array)